# Unidad 6: Optimización y rendimiento en Spark

**Tecnicatura en Datos – Procesamiento con Apache Spark (Databricks)**  
Unidad 6 de 10 — Duración estimada: 2:30 hs

> **Nota Databricks:** `spark` ya está disponible. La Spark UI está accesible desde la pestaña **Spark** del clúster. Todos los `explain()` son más informativos ahí.

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, spark_partition_id, count, broadcast
import time
print("Imports OK | Spark", spark.version)

---
## 1. Catalyst Optimizer y Tungsten

Spark no ejecuta el código como lo escribís. Antes de ejecutar, aplica dos capas de optimización:

| Componente | Qué optimiza | Ejemplo |
|---|---|---|
| **Catalyst** | Plan lógico y físico | Mueve filtros antes de joins (predicate pushdown) |
| **Tungsten** | Memoria y CPU | Code generation, off-heap memory |

```
Tu código
   ↓
Plan lógico no resuelto
   ↓  (Catalyst: resolución de nombres y tipos)
Plan lógico resuelto
   ↓  (Catalyst: optimizaciones: predicate pushdown, column pruning...)
Plan lógico optimizado
   ↓  (Catalyst: genera múltiples planes físicos y elige el mejor)
Plan físico
   ↓  (Tungsten: genera bytecode optimizado)
Ejecución
```

---
## 2. explain() — Ver el plan de ejecución

### Ejemplo 1 — Simple: plan físico básico

In [ ]:
data = [(i, "producto_%d" % i, float(i * 100)) for i in range(1, 11)]
df = spark.createDataFrame(data, ["id", "producto", "monto"])

operacion = df.filter(col("monto") > 500).select("producto", "monto")

print("=== Plan físico (simplificado) ===")
operacion.explain()

**Resultado esperado:**
```
== Physical Plan ==
*(1) Project [producto#12, monto#13]
+- *(1) Filter (monto#13 > 500.0)
   +- *(1) Scan ExistingRDD
```

> El `*` indica **Whole-Stage Code Generation** (Tungsten). `Filter` aparece antes de `Project`: Catalyst aplicó predicate pushdown.

### Ejemplo 2 — Medio: detectar shuffles con `explain(True)`

In [ ]:
df_ventas = spark.createDataFrame([
    ("AR", 1500.0), ("MX", 800.0), ("AR", 200.0),
    ("CL", 450.0),  ("AR", 900.0), ("MX", 300.0),
], ["pais", "monto"])

resumen = df_ventas.groupBy("pais").agg(spark_sum("monto").alias("total"))

print("=== Plan completo (lógico + optimizado + físico) ===")
resumen.explain(True)

**Buscar en el plan físico:**
```
+- HashAggregate(...)             ← agregación final post-shuffle
   +- Exchange hashpartitioning(pais, 200)   ← SHUFFLE aquí
      +- HashAggregate(...)       ← agregación parcial pre-shuffle
```

> Dos `HashAggregate`: el primero agrega localmente por partición (reduce datos), el segundo combina post-shuffle. El `Exchange` es el shuffle.

### Ejemplo 3 — Avanzado: comparar SortMergeJoin vs BroadcastHashJoin

In [ ]:
df_grande = spark.range(1, 100001).withColumnRenamed("id", "cliente_id") \
    .withColumn("monto", (col("cliente_id") * 1.5))

df_chico = spark.createDataFrame([(1, "AR"), (2, "MX"), (3, "CL")], ["cliente_id", "pais"])

# Deshabilitar broadcast automático para ver el SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("=== SIN broadcast (SortMergeJoin — dos shuffles) ===")
df_grande.join(df_chico, "cliente_id", "inner").explain()

print("\n=== CON broadcast (BroadcastHashJoin — sin shuffle) ===")
df_grande.join(broadcast(df_chico), "cliente_id", "inner").explain()

# Restaurar el umbral
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

**Resultado esperado:**
```
=== SIN broadcast ===
SortMergeJoin [cliente_id], [cliente_id], Inner
:- Sort ... +- Exchange hashpartitioning(...)   ← shuffle
+- Sort ... +- Exchange hashpartitioning(...)   ← shuffle

=== CON broadcast ===
BroadcastHashJoin [cliente_id], [cliente_id], Inner, BuildRight
:- Range (1, 100001, ...)
+- BroadcastExchange HashedRelationBroadcastMode   ← sin shuffle
```

---
## 3. Particionamiento

### Ejemplo 1 — Simple: ver particiones y reducir con `coalesce`

In [ ]:
df = spark.range(1, 1_000_001)

print("Particiones originales:", df.rdd.getNumPartitions())

# coalesce: reduce sin shuffle (eficiente para escritura)
df_reducido = df.coalesce(4)
print("Particiones tras coalesce(4):", df_reducido.rdd.getNumPartitions())

# repartition: shuffle completo (redistribuye datos)
df_repartido = df.repartition(8)
print("Particiones tras repartition(8):", df_repartido.rdd.getNumPartitions())

**Resultado esperado:**
```
Particiones originales:     8     # varía según cores del clúster
Particiones tras coalesce:  4     # sin shuffle
Particiones tras repartition: 8   # con shuffle
```

### Ejemplo 2 — Medio: detectar y corregir skew con `repartition`

In [ ]:
# Crear DataFrame desbalanceado (skew)
data_skewed = [("AR",)] * 9000 + [("MX",)] * 500 + [("CL",)] * 500
df_skewed = spark.createDataFrame(data_skewed, ["pais"])

print("=== ANTES de repartition (skew) ===")
df_skewed.groupBy(spark_partition_id().alias("particion")).count() \
    .orderBy("particion").show()

# repartition por columna: usa hash para distribuir uniformemente
df_balanced = df_skewed.repartition(6, "pais")

print("=== DESPUÉS de repartition(6, 'pais') ===")
df_balanced.groupBy(spark_partition_id().alias("particion")).count() \
    .orderBy("particion").show()

**Resultado esperado:**
```
=== ANTES ===
+---------+-----+
|particion|count|
+---------+-----+
|        0| 9000|   ← skew: 90% de los datos en una partición
|        1|  500|
|        2|  500|
+---------+-----+

=== DESPUÉS ===
+---------+-----+
|particion|count|
+---------+-----+
|        0| 3000|   ← distribuido uniformemente
|        1| 3000|
|        2| 3000|
|        3|  167|
|        4|  167|
|        5|  166|
+---------+-----+
```

| Operación | Shuffle | Cuándo usar |
|---|---|---|
| `coalesce(n)` | No | Reducir antes de escribir |
| `repartition(n)` | Sí | Balancear antes de joins/aggregations |

---
## 4. Shuffles — qué son y cómo reducirlos

In [ ]:
# Operaciones que SÍ generan shuffle
df_test = spark.range(1, 10001).withColumn("grupo", col("id") % 5)

print("Operaciones con shuffle:")
print("  groupBy + agg → shuffle")
print("  distinct      → shuffle")
print("  orderBy       → shuffle")
print("  join (sin broadcast) → shuffle")
print()

# Ver cuántos shuffles genera esta cadena
operacion = df_test.groupBy("grupo").count().orderBy("grupo")
operacion.explain()

# Ver el número de particiones de shuffle configurado
print(f"\nspark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")
print("(reducir este valor en datasets pequeños puede mejorar el rendimiento)")

In [ ]:
# Para datasets pequeños o medianos: reducir shuffle.partitions evita crear 200 particiones vacías
spark.conf.set("spark.sql.shuffle.partitions", "8")

t0 = time.time()
df_test.groupBy("grupo").count().collect()
t1 = time.time()
print(f"Con 8 shuffle partitions:   {t1-t0:.3f}s")

spark.conf.set("spark.sql.shuffle.partitions", "200")  # restaurar
t2 = time.time()
df_test.groupBy("grupo").count().collect()
t3 = time.time()
print(f"Con 200 shuffle partitions: {t3-t2:.3f}s")

---
## 5. Caché y persistencia

In [ ]:
from pyspark import StorageLevel

df_base = spark.range(1, 1_000_001) \
    .withColumn("grupo", col("id") % 100) \
    .withColumn("valor", (col("id") * 2.5))

# SIN caché: Spark recalcula df_base dos veces
t0 = time.time()
conteo = df_base.count()
total  = df_base.agg({"valor": "sum"}).collect()[0][0]
t1 = time.time()
print(f"Sin caché:   {t1-t0:.3f}s  (recalcula 2 veces)")

# CON caché: df_base se calcula una vez y se guarda en RAM
df_base.cache()
df_base.count()  # materializa el caché

t2 = time.time()
conteo = df_base.count()
total  = df_base.agg({"valor": "sum"}).collect()[0][0]
t3 = time.time()
print(f"Con caché:   {t3-t2:.3f}s  (usa datos ya calculados)")

df_base.unpersist()  # liberar memoria
print(f"\nConteo: {conteo:,} | Total valor: {total:,.1f}")

**Resultado esperado (orientativo):**
```
Sin caché:   1.842s  (recalcula 2 veces)
Con caché:   0.312s  (usa datos ya calculados)

Conteo: 1,000,000 | Total valor: 1,250,000,500.0
```

### ¿Cuándo usar caché?

| Situación | Usar caché |
|---|---|
| DataFrame usado 2+ veces en el pipeline | ✅ Sí |
| Algoritmos iterativos (ML) | ✅ Sí |
| DataFrame grande usado solo una vez | ❌ No (consume RAM innecesariamente) |
| DataFrame que cambia frecuentemente | ❌ No |

---
## 6. Broadcast Joins

### Ejemplo 1 — Simple: broadcast de tabla de códigos

In [ ]:
df_tx = spark.range(1, 100001) \
    .withColumn("pais_id", (col("id") % 3).cast("int")) \
    .withColumn("monto",   (col("id") * 1.5))

df_paises = spark.createDataFrame([
    (0, "Argentina"),
    (1, "México"),
    (2, "Chile"),
], ["pais_id", "nombre_pais"])

# Con broadcast: df_paises (3 filas) viaja a cada executor
resultado = df_tx.join(broadcast(df_paises), "pais_id", "inner")
resultado.show(5)

**Resultado esperado:**
```
+-------+---+------+-----------+
|pais_id| id| monto|nombre_pais|
+-------+---+------+-----------+
|      1|  1|   1.5|     México|
|      2|  2|   3.0|      Chile|
|      0|  3|   4.5|  Argentina|
+-------+---+------+-----------+
```

### Ejemplo 2 — Medio: broadcast automático con threshold

In [ ]:
# Ver el umbral actual (por defecto 10 MB)
threshold_actual = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"Umbral actual: {int(threshold_actual) / 1024 / 1024:.1f} MB")

# Aumentar a 50 MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(50 * 1024 * 1024))

# Join SIN broadcast() explícito — Spark decide solo
resultado_auto = df_tx.join(df_paises, "pais_id", "inner")
resultado_auto.explain()

# Restaurar
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", threshold_actual)

**Buscar en el plan:**
```
BroadcastHashJoin [pais_id], [pais_id], Inner, BuildRight
:- Project [pais_id, id, monto]
+- BroadcastExchange HashedRelationBroadcastMode   ← automático
```

### Ejemplo 3 — Avanzado: comparar tiempos con y sin broadcast

In [ ]:
df_grande = spark.range(1, 2_000_001) \
    .withColumn("pais_id", (col("id") % 3).cast("int")) \
    .withColumn("monto",   (col("id") * 2.0))

df_dim = spark.createDataFrame([
    (0, "Argentina", "ARS"),
    (1, "México",    "MXN"),
    (2, "Chile",     "CLP"),
], ["pais_id", "nombre", "moneda"])

# Deshabilitar broadcast automático para medir diferencia real
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Sin broadcast
t0 = time.time()
df_grande.join(df_dim, "pais_id", "inner").agg({"monto": "sum"}).collect()
t1 = time.time()
print(f"Sin broadcast: {t1 - t0:.2f}s")

# Con broadcast
t2 = time.time()
df_grande.join(broadcast(df_dim), "pais_id", "inner").agg({"monto": "sum"}).collect()
t3 = time.time()
print(f"Con broadcast: {t3 - t2:.2f}s")

mejora = (t1 - t0) / (t3 - t2)
print(f"Mejora: {mejora:.1f}x más rápido")

# Restaurar
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

**Resultado esperado (orientativo):**
```
Sin broadcast: 12.43s
Con broadcast:  2.18s
Mejora: 5.7x más rápido
```

---
## 7. Práctica guiada

### Ejercicio 1 — Simple: forzar SortMergeJoin y leer el plan

In [ ]:
df_tx_e = spark.range(1, 50001) \
    .withColumn("pais_id", (col("id") % 3).cast("int")) \
    .withColumn("monto",   (col("id") * 1.5))

df_paises_e = spark.createDataFrame([
    (0, "Argentina"), (1, "México"), (2, "Chile")
], ["pais_id", "pais"])

# Deshabilitar broadcast automático → fuerza SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("=== Plan con SortMergeJoin ===")
df_tx_e.join(df_paises_e, "pais_id", "inner").explain()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

**Resultado esperado:**
```
SortMergeJoin [pais_id], [pais_id], Inner
:- Sort ... +- Exchange hashpartitioning(pais_id, 200)   ← shuffle
+- Sort ... +- Exchange hashpartitioning(pais_id, 200)   ← shuffle
```

### Ejercicio 2 — Avanzado: aplicar broadcast y comparar planes y tiempos

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # sin auto-broadcast

# Sin broadcast
t0 = time.time()
df_tx_e.join(df_paises_e, "pais_id", "inner").count()
t1 = time.time()
print(f"Sin broadcast: {t1 - t0:.3f}s")

# Con broadcast explícito
t2 = time.time()
df_tx_e.join(broadcast(df_paises_e), "pais_id", "inner").count()
t3 = time.time()
print(f"Con broadcast: {t3 - t2:.3f}s")

print("\n=== Plan con BroadcastHashJoin ===")
df_tx_e.join(broadcast(df_paises_e), "pais_id", "inner").explain()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

**Resultado esperado:**
```
Sin broadcast: 3.412s
Con broadcast: 0.743s

=== Plan con BroadcastHashJoin ===
BroadcastHashJoin [pais_id], [pais_id], Inner, BuildRight
:- Project [pais_id, id, monto]
+- BroadcastExchange HashedRelationBroadcastMode   ← sin Exchange en tabla grande
```

---
## 8. Resumen de reglas de optimización

```
┌─────────────────────────────────────────────────────────────┐
│                 Checklist de optimización Spark              │
├─────────────────────────────────────────────────────────────┤
│ ✅ Usar explain() para entender el plan antes de producción  │
│ ✅ Filtrar ANTES del join (menos datos a mover)              │
│ ✅ Usar broadcast() en joins con tablas < 10-50 MB           │
│ ✅ Usar coalesce() antes de escribir a disco                 │
│ ✅ Usar repartition() para corregir skew                     │
│ ✅ cache() solo si el DF se usa 2+ veces                    │
│ ✅ unpersist() cuando el caché ya no es necesario            │
│ ✅ Reducir shuffle.partitions para datasets pequeños         │
│ ✅ Usar schema explícito en producción (no inferSchema)      │
│ ❌ Evitar UDFs si existe función nativa equivalente          │
│ ❌ Evitar collect() sobre datasets grandes                   │
└─────────────────────────────────────────────────────────────┘
```

---
## 9. Preguntas de revisión

1. ¿Qué optimizaciones aplica Catalyst automáticamente?
2. ¿Qué diferencia hay entre `coalesce` y `repartition`?
3. ¿Por qué los shuffles son costosos?
4. ¿Cuándo conviene usar `cache()` y cuándo no?
5. ¿Qué condición debe cumplir una tabla para usar broadcast join?

---
**Próxima unidad:** Lectura y escritura de datos (Delta Lake, Parquet, CSV)